In [ ]:
import os

REPO = 'giomamaca/walmart-sales-forecasting'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import userdata

    github_token = userdata.get('GITHUB_TOKEN')
    repo_dir = f'/content/{REPO.split("/")[1]}'
    if not os.path.exists(repo_dir):
        !git clone -q https://{github_token}@github.com/{REPO}.git {repo_dir}
    os.chdir(repo_dir)

    !pip install -q mlflow

In [ ]:
import mlflow

!mlflow db upgrade sqlite:///mlflow.db
mlflow.set_tracking_uri('sqlite:///mlflow.db')

In [ ]:
ARCHS = ['XGBoost', 'LightGBM', 'Prophet', 'DLinear']

client = mlflow.MlflowClient()

def cv_wmae(arch):
    exp = client.get_experiment_by_name(f'{arch}_Training')
    if exp is None:
        return None
    runs = client.search_runs(exp.experiment_id, filter_string=f"attributes.run_name = '{arch}_CV'",
                              order_by=['attributes.start_time DESC'], max_results=1)
    return runs[0].data.metrics.get('wmae_mean') if runs else None

scores = {arch: score for arch in ARCHS if (score := cv_wmae(arch)) is not None}
best_arch = min(scores, key=scores.get)
scores, best_arch

In [ ]:
MODEL_NAME = 'walmart-sales-model'

exp = client.get_experiment_by_name(f'{best_arch}_Training')
final_run = client.search_runs(exp.experiment_id, filter_string=f"attributes.run_name = '{best_arch}_Final'",
                               order_by=['attributes.start_time DESC'], max_results=1)[0]

registered = [m.name for m in client.search_registered_models()]
current = client.get_registered_model(MODEL_NAME).latest_versions[0] if MODEL_NAME in registered else None
if current is None or current.run_id != final_run.info.run_id:
    mlflow.register_model(f'runs:/{final_run.info.run_id}/model', MODEL_NAME)

client.get_registered_model(MODEL_NAME).latest_versions[0]

In [ ]:
if IN_COLAB:
    import requests
    gh_user = requests.get('https://api.github.com/user', headers={'Authorization': f'token {github_token}'}).json()['login']
    !git config user.name "{gh_user}"
    !git config user.email "{gh_user}@users.noreply.github.com"

    !git add mlflow.db mlruns
    !git commit -m "Register best model"
    !git push